# Ensemble Methods — AdaBoost (Adaptive Boosting)

**Boosting** trains weak learners **sequentially**, each one focusing on the mistakes of the previous ones.
AdaBoost (Freund & Schapire, 1997) with labels $y_i\in\{-1,+1\}$:

1. Initialise sample weights $w_i = 1/n$.
2. For $m = 1..M$:
   * Fit a weak learner $h_m$ (here a **decision stump**) minimising weighted error $\varepsilon_m = \sum_i w_i\,[h_m(x_i)\ne y_i]$
   * Learner weight: $\alpha_m = \frac{1}{2}\ln\frac{1-\varepsilon_m}{\varepsilon_m}$
   * Update: $w_i \leftarrow w_i\, e^{-\alpha_m y_i h_m(x_i)}$, then normalise. Misclassified points gain weight.
3. Final classifier: $H(x) = \text{sign}\Big(\sum_m \alpha_m h_m(x)\Big)$

AdaBoost is equivalent to forward stagewise minimisation of the **exponential loss** $\sum_i e^{-y_iF(x_i)}$,
and its training error is bounded by $\prod_m 2\sqrt{\varepsilon_m(1-\varepsilon_m)}$, which shrinks exponentially as long as each
$\varepsilon_m < 0.5$.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

## Weighted decision stump

In [ ]:
class DecisionStump:
    """One-level tree: predict `polarity` if x[feature] > threshold else -polarity."""

    def fit(self, X, y, w):
        n, d = X.shape
        best_err = np.inf
        for f in range(d):
            order = np.argsort(X[:, f])
            xs, ys, ws = X[order, f], y[order], w[order]
            # split k: points 0..k-1 go left. Error of "left=-1, right=+1" for every k at once:
            left_pos = np.r_[0, np.cumsum(ws * (ys == 1))]
            left_neg = np.r_[0, np.cumsum(ws * (ys == -1))]
            err = left_pos + (left_neg[-1] - left_neg)          # positives on left + negatives on right
            thresholds = np.r_[xs[0] - 1, (xs[:-1] + xs[1:]) / 2, xs[-1] + 1]
            valid = np.r_[True, xs[1:] != xs[:-1], True]
            for polarity, e in [(1, err), (-1, w.sum() - err)]:   # flipped stump has complementary error
                e = np.where(valid, e, np.inf)
                k = np.argmin(e)
                if e[k] < best_err:
                    best_err = e[k]
                    self.feature, self.threshold, self.polarity = f, thresholds[k], polarity
        return self

    def predict(self, X):
        return np.where(X[:, self.feature] > self.threshold, self.polarity, -self.polarity)

## AdaBoost

In [ ]:
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators

    def fit(self, X, y):
        y = np.where(y <= 0, -1, 1)
        n = len(y)
        w = np.full(n, 1 / n)
        self.stumps, self.alphas, self.errors, self.weight_history = [], [], [], [w.copy()]
        for _ in range(self.n_estimators):
            stump = DecisionStump().fit(X, y, w)
            pred = stump.predict(X)
            eps = np.clip(w[pred != y].sum(), 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - eps) / eps)
            w = w * np.exp(-alpha * y * pred)
            w /= w.sum()
            self.stumps.append(stump); self.alphas.append(alpha); self.errors.append(eps)
            self.weight_history.append(w.copy())
        return self

    def decision_function(self, X, n_estimators=None):
        m = n_estimators or len(self.stumps)
        return sum(a * s.predict(X) for a, s in zip(self.alphas[:m], self.stumps[:m]))

    def staged_decision_function(self, X):
        F = np.zeros(len(X))
        for a, s in zip(self.alphas, self.stumps):
            F = F + a * s.predict(X)
            yield F

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)

## Data: a circular boundary (impossible for a single stump)
Label = 1 if the point lies outside the circle $\lVert x\rVert^2 > \chi^2_2(0.5)\approx 1.386$.

In [ ]:
rng = np.random.RandomState(1)
X = rng.normal(size=(600, 2))
y = ((X ** 2).sum(1) > 1.386).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

ada = AdaBoost(n_estimators=300).fit(X_train, y_train)
print("Single stump test acc:", accuracy_score(y_test, AdaBoost(1).fit(X_train, y_train).predict(X_test)))
print("AdaBoost(300) test acc:", accuracy_score(y_test, ada.predict(X_test)))

In [ ]:
def plot_boundary(model, X, y, n_est, ax):
    xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 200), np.linspace(-3.5, 3.5, 200))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()], n_est).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(f"{n_est} stump(s)")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, m in zip(axes, [1, 5, 30, 300]):
    plot_boundary(ada, X_train, y_train, m, ax)
plt.tight_layout(); plt.show()

## Training / test error and the theoretical bound

In [ ]:
y_tr_pm, y_te_pm = np.where(y_train == 1, 1, -1), np.where(y_test == 1, 1, -1)
train_err = [np.mean(np.sign(F) != y_tr_pm) for F in ada.staged_decision_function(X_train)]
test_err = [np.mean(np.sign(F) != y_te_pm) for F in ada.staged_decision_function(X_test)]
eps = np.array(ada.errors)
bound = np.cumprod(2 * np.sqrt(eps * (1 - eps)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(train_err, label="train error"); axes[0].plot(test_err, label="test error")
axes[0].plot(bound, "k--", label=r"bound $\prod 2\sqrt{\epsilon(1-\epsilon)}$")
axes[0].set_xlabel("boosting round"); axes[0].set_ylabel("error"); axes[0].set_ylim(0, 0.6); axes[0].legend()
axes[0].set_title("Error vs number of stumps")
axes[1].plot(eps, label=r"weighted error $\epsilon_m$"); axes[1].plot(ada.alphas, label=r"$\alpha_m$")
axes[1].axhline(0.5, ls=":", c="gray"); axes[1].set_xlabel("boosting round"); axes[1].legend()
axes[1].set_title("Weak learner error and weight")
plt.tight_layout(); plt.show()

## How the sample weights evolve
Marker size ∝ weight. Points near the boundary that are repeatedly misclassified become dominant.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, m in zip(axes, [0, 3, 50]):
    w = ada.weight_history[m]
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=3000 * w, edgecolors="k", alpha=0.7)
    ax.set_title(f"sample weights after {m} rounds")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

## Summary
* AdaBoost reduces **bias**: a sum of very simple stumps forms a complex boundary.
* Weights force later learners to concentrate on hard examples.
* Sensitive to label noise/outliers (they get huge weights); in practice keep $M$ moderate or use a learning rate.
* Multiclass extension: **SAMME**, which uses $\alpha_m = \ln\frac{1-\varepsilon_m}{\varepsilon_m} + \ln(K-1)$.